In [29]:
import numpy 

In [30]:
import numpy as np

np.random.randint(10, 120)

105

In [15]:
import yaml
import pandas as pd
import os
import sys

current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, project_root)

In [16]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [17]:
from functions.ann_model import KerasBinaryClassifier
from functions.evaluate_model import evaluate_model

In [18]:
from utils.utils import to_jsonl

In [19]:
# 1. Carregar configurações
with open(os.path.join("../config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

# pipeline selection    
with open(os.path.join("../config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)

# model selection    
with open(os.path.join( "../config/model.yaml"), "r") as f:
    config_model = yaml.safe_load(f)

In [20]:
# Get feature eng data
pipeline_name = "Pipeline3"

X_train = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_train_feat_eng_{pipeline_name}.parquet")
)

y_train = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_train_feat_eng_{pipeline_name}.parquet")
)

X_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_val_feat_eng_{pipeline_name}.parquet")
)

y_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_val_feat_eng_{pipeline_name}.parquet")
)

In [21]:
X_val.drop(
    columns = config_model['single_model']['cols_2_drop'],
    inplace=True
) 

X_train.drop(
    columns = config_model['single_model']['cols_2_drop'],
    inplace=True
) 

In [22]:
path = r"C:\Users\gustavo\Documents\Data Science\08-GitHub\kaggle\Classification\Titanic\models\single_model\predictions\X_val_proba_LogisticRegression.parquet"

In [24]:
tmp = pd.read_parquet(path)

In [27]:
pd.concat([tmp, y_val], axis=1)

,probability,Survived
709,0.308128,1
439,0.157775,0
840,0.109637,0
720,0.821371,1
39,0.502767,1
...,...,...
433,0.109637,0
773,0.109637,0
25,0.263044,1
84,0.686751,1


In [26]:
y_val

,Survived
709,1
439,0
840,0
720,1
39,1
...,...
433,0
773,0
25,1
84,1


In [25]:
tmp

,probability
709,0.308128
439,0.157775
840,0.109637
720,0.821371
39,0.502767
...,...
433,0.109637
773,0.109637
25,0.263044
84,0.686751


In [8]:
from functions.cross_validate import cross_validate_kfold, cross_validate_StratifiedKFold, cross_validate_stratified_gs

In [9]:
from functions.model_selection import grid_search_single_model_StratifiedKFold, grid_search_single_model_kfold

In [10]:
from functions.single_model import SingleModelOrchestrator

In [11]:
model_name = "LogisticRegression"

In [ ]:
model_orchestrator = SingleModelOrchestrator()
model_config = model_orchestrator.apply(model_name)  
    
param_grid = {
    "penalty":['l2'],
    "C":[10, 1, 0.1], 
    "max_iter":[200, 300, 1000], 
    "solver":['liblinear', 'saga', 'lbfgs']    
    }     


In [16]:
best_paramns = grid_search_single_model_StratifiedKFold(
        X_train, 
        y_train, 
        model_config['model'], 
        param_grid, 
        metric='roc_auc'
        )     

In [17]:

# 4. train model 
    # set params
model_clf = model_config['model'].set_params(**best_paramns)


In [19]:
    # cross-validade
df_cv = cross_validate_StratifiedKFold(
    X_train, 
    y_train, 
    model_clf,
    model_config
    )

In [20]:
df_cv

(   fold  train_score  test_score    model_type               model  \
 0     0       0.8207      0.8741  single_model  LogisticRegression   
 1     1       0.8243      0.8462  single_model  LogisticRegression   
 2     2       0.8281      0.8099  single_model  LogisticRegression   
 3     3       0.8404      0.7606  single_model  LogisticRegression   
 4     4       0.8193      0.8028  single_model  LogisticRegression   
 
                     timestamp  
 0  2026-04-08T12:41:35.104990  
 1  2026-04-08T12:41:35.104990  
 2  2026-04-08T12:41:35.104990  
 3  2026-04-08T12:41:35.104990  
 4  2026-04-08T12:41:35.104990  ,
 LogisticRegression(C=0.1, max_iter=200, random_state=23, solver='saga'))

In [ ]:
model = KerasBinaryClassifier(input_dim=X_train.shape[1], epochs=70)

clf = model.fit(X_train, y_train)

In [ ]:
metrics_train = evaluate_model(clf, X_train, y_train)

In [ ]:
metrics_val = evaluate_model(clf, X_val, y_val)

In [ ]:
metrics_train['accuracy_score']

In [ ]:
metrics_val['accuracy_score']

In [ ]:
pipeline = make_pipeline(
    KerasBinaryClassifier(input_dim=X_train.shape[1])
)

pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "kerasbinaryclassifier__epochs": [70, 90],
    "kerasbinaryclassifier__batch_size": [16, 32],
    "kerasbinaryclassifier__hidden_units": [(32,16), (64,32), (32,32)],
    "kerasbinaryclassifier__learning_rate": [0.001, 0.0005]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

In [ ]:
grid.best_params_

In [ ]:
metrics_train_grid = evaluate_model(grid, X_train, y_train)

In [ ]:
metrics_train_grid['accuracy_score']

In [ ]:
metrics_val_grid = evaluate_model(grid, X_val, y_val)

In [ ]:
metrics_val_grid['accuracy_score']